# Práctica 1: Perceptrón multicapa.

Tu jefe pidió a RH que recolectara datos de desempeño de tus compañeros, los resultados se almacenaron en un csv. El punto critico de estos datos es la satisfacción del empleado, entonces ¿Podremos estimar la satisfacción de los empleados con los datos recabados?.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers, models

df = pd.read_csv('Extended_Employee_Performance_and_Productivity_Data.csv')
df.info()

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Filtrar las columnas numéricas
numeric_columns = df.select_dtypes(include=['number']).drop('Employee_ID',axis=1)


# Si numeric_columns es un Index, conviértelo a lista
cols = list(numeric_columns)

fig, axes = plt.subplots(1, len(cols), figsize=(5 * len(cols), 4))

for i, col in enumerate(cols):
    axes[i].hist(df[col], bins=20, color='skyblue', edgecolor='black')
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

**Problemas**, tenemos distribuciones con picos, esos nos indica categorías. Por otro lado, tenemos variables con "valles" en su distribución (distribuciones multimodales) por lo que resultaría óptimo aplicar técnicas de feature engeneering. Por último tenemos distribuciones uniformes, por lo que cada una requeriría un procesamiento indivudual, hagamos la vista gorda e intentemos ajustar un MLP con estos datos, solo estandaricemos nuestros datos.

---

## Implementación de Red:

To**memos los datos numéricos como nuestra variable X, y la variable objetivo como ***'Employee_Satisfaction_Score'***.
- **Actividad 1**: Para todos los strings ``'@modif@'`` que aparescan en el siguiente bloque de código cámbialos para que el código funcione.

In [6]:
X = numeric_columns.drop('Employee_Satisfaction_Score',axis = 1)
y = numeric_columns['Employee_Satisfaction_Score']
y = y.apply(lambda x: round(x)-1) #Cambiamos la variable objetivo a 5 categorías numéricas

scaler = StandardScaler()
X_standar = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_standar, y, test_size=0.33, random_state=42)

y_onehot_train = tf.keras.utils.to_categorical(y_train, 5)
y_onehot_test = tf.keras.utils.to_categorical(y_test,5)

NameError: name 'numeric_columns' is not defined

- **Actividad 2:** Implementa 3 arquitecturas de MLP, cada una con su propio nombre, cambiando la estructura de dichas arquitecturas (capas, neuronas por capa, función de activación, etc). 

In [ ]:
# Arquitectura 1: MLP Simple con pocas capas
modelo_simple = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(5, activation='softmax')
])

# Arquitectura 2: MLP Profundo con más capas
modelo_profundo = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(16, activation='relu'),
    layers.Dense(5, activation='softmax')
])

# Arquitectura 3: MLP con diferentes funciones de activación
modelo_hibrido = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(96, activation='tanh'),
    layers.Dropout(0.25),
    layers.Dense(48, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(24, activation='sigmoid'),
    layers.Dense(5, activation='softmax')
])

print("Arquitecturas creadas:")
print("1. Modelo Simple: 64->32->5 neuronas con ReLU")
print("2. Modelo Profundo: 128->64->32->16->5 neuronas con ReLU")
print("3. Modelo Híbrido: 96->48->24->5 neuronas con tanh, relu, sigmoid")


NameError: name 'models' is not defined

- **Actividad 3:** Compila y ajusta tus tres modelos con sus respectivos hiperparámetros.

# Compilación y entrenamiento del Modelo Simple
print("=== ENTRENANDO MODELO SIMPLE ===")
modelo_simple.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

historia_simple = modelo_simple.fit(
    X_train, y_onehot_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_onehot_test),
    verbose=1
)

# Compilación y entrenamiento del Modelo Profundo
print("\n=== ENTRENANDO MODELO PROFUNDO ===")
modelo_profundo.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

historia_profundo = modelo_profundo.fit(
    X_train, y_onehot_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_onehot_test),
    verbose=1
)

# Compilación y entrenamiento del Modelo Híbrido
print("\n=== ENTRENANDO MODELO HÍBRIDO ===")
modelo_hibrido.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

historia_hibrido = modelo_hibrido.fit(
    X_train, y_onehot_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_onehot_test),
    verbose=1
)


In [3]:
# Evaluación y comparación de modelos
print("=== EVALUACIÓN DE MODELOS ===")

# Evaluar modelo simple
test_loss_simple, test_acc_simple = modelo_simple.evaluate(X_test, y_onehot_test, verbose=0)
print(f"Modelo Simple - Pérdida: {test_loss_simple:.4f}, Precisión: {test_acc_simple:.4f}")

# Evaluar modelo profundo
test_loss_profundo, test_acc_profundo = modelo_profundo.evaluate(X_test, y_onehot_test, verbose=0)
print(f"Modelo Profundo - Pérdida: {test_loss_profundo:.4f}, Precisión: {test_acc_profundo:.4f}")

# Evaluar modelo híbrido
test_loss_hibrido, test_acc_hibrido = modelo_hibrido.evaluate(X_test, y_onehot_test, verbose=0)
print(f"Modelo Híbrido - Pérdida: {test_loss_hibrido:.4f}, Precisión: {test_acc_hibrido:.4f}")

# Visualización de las curvas de entrenamiento
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Pérdida del modelo simple
axes[0,0].plot(historia_simple.history['loss'], label='Entrenamiento')
axes[0,0].plot(historia_simple.history['val_loss'], label='Validación')
axes[0,0].set_title('Modelo Simple - Pérdida')
axes[0,0].set_xlabel('Época')
axes[0,0].set_ylabel('Pérdida')
axes[0,0].legend()

# Precisión del modelo simple
axes[1,0].plot(historia_simple.history['accuracy'], label='Entrenamiento')
axes[1,0].plot(historia_simple.history['val_accuracy'], label='Validación')
axes[1,0].set_title('Modelo Simple - Precisión')
axes[1,0].set_xlabel('Época')
axes[1,0].set_ylabel('Precisión')
axes[1,0].legend()

# Pérdida del modelo profundo
axes[0,1].plot(historia_profundo.history['loss'], label='Entrenamiento')
axes[0,1].plot(historia_profundo.history['val_loss'], label='Validación')
axes[0,1].set_title('Modelo Profundo - Pérdida')
axes[0,1].set_xlabel('Época')
axes[0,1].set_ylabel('Pérdida')
axes[0,1].legend()

# Precisión del modelo profundo
axes[1,1].plot(historia_profundo.history['accuracy'], label='Entrenamiento')
axes[1,1].plot(historia_profundo.history['val_accuracy'], label='Validación')
axes[1,1].set_title('Modelo Profundo - Precisión')
axes[1,1].set_xlabel('Época')
axes[1,1].set_ylabel('Precisión')
axes[1,1].legend()

# Pérdida del modelo híbrido
axes[0,2].plot(historia_hibrido.history['loss'], label='Entrenamiento')
axes[0,2].plot(historia_hibrido.history['val_loss'], label='Validación')
axes[0,2].set_title('Modelo Híbrido - Pérdida')
axes[0,2].set_xlabel('Época')
axes[0,2].set_ylabel('Pérdida')
axes[0,2].legend()

# Precisión del modelo híbrido
axes[1,2].plot(historia_hibrido.history['accuracy'], label='Entrenamiento')
axes[1,2].plot(historia_hibrido.history['val_accuracy'], label='Validación')
axes[1,2].set_title('Modelo Híbrido - Precisión')
axes[1,2].set_xlabel('Época')
axes[1,2].set_ylabel('Precisión')
axes[1,2].legend()

plt.tight_layout()
plt.show()

# Resumen de resultados
print("\n=== RESUMEN DE RESULTADOS ===")
resultados = {
    'Modelo Simple': test_acc_simple,
    'Modelo Profundo': test_acc_profundo,
    'Modelo Híbrido': test_acc_hibrido
}

mejor_modelo = max(resultados, key=resultados.get)
print(f"El mejor modelo es: {mejor_modelo} con una precisión de {resultados[mejor_modelo]:.4f}")


=== EVALUACIÓN DE MODELOS ===


NameError: name 'modelo_simple' is not defined

- **Actividad 4:** Sube tus cambios al repositorio, envía el link de tu repositorio a la actividad 2 de tu checkpoint 2 y contesta las preguntas de dicha actividad.